In [0]:
from pyspark.sql.functions import col
import pyspark.sql.functions as F
import pandas as pd
import os

In [0]:
df_comex_mun = spark.read.table("obsref.comex.flat_municipio")
df_comex_ncm = spark.read.table("obsref.comex.flat_ncm")
df_dic = pd.read_csv("https://docs.google.com/spreadsheets/d/e/2PACX-1vQ5AE-EGJxoKed9vxCBEeFAf6LM6GRdNvWhY4npQLkHDk9muuRM6f-Tn4lcdHtd5R9CsA4TlWxn5XDu/pub?gid=1951449408&single=true&output=csv")

In [0]:
# ------------------------- Tratando dados da base estadual (NCM)
df_comex_ncm = df_comex_ncm.filter(
    (col("nr_ano").cast("int") >= 2020) &
    (col("sg_uf") == "SC")
)

df_comex_ncm = df_comex_ncm.select(
    "nr_ano", "nr_mes", "tp_carga", "cd_sh4", "cd_pais", "ds_pais", "vl_fob"
)

df_dic = spark.createDataFrame(df_dic)

df_dic_final = df_dic.select(
    "cd_sh4", "nm_produto", "nm_sc_competitiva", "ds_cnae_divisao", "ds_cnae_grupo").groupBy("cd_sh4").agg(
    F.first("nm_produto").alias("nm_produto"),
    F.first("nm_sc_competitiva").alias("nm_sc_competitiva"),
    F.first("ds_cnae_divisao").alias("ds_cnae_divisao"),
    F.first("ds_cnae_grupo").alias("ds_cnae_grupo")
)

df_comex_ncm = df_comex_ncm.join(
    F.broadcast(df_dic_final), 
    on="cd_sh4", 
    how="left"
)

In [0]:
output = 

df_comex_ncm.write.mode("overwrite").parquet("/Workspace/Observatorio_dev/painel_comex/teste")